# WikiGPT 01: Prepare data

Downloads English Wikipedia shards, trains the 8192-token BPE tokenizer and writes `train.bin`, `val.bin`, `tokenizer.json` and `meta.json` to `/kaggle/working/data`.

**Settings:** Accelerator None, Internet On.  
**Inputs:** the `wikigpt-code` dataset.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch

INPUT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")


def find_input(pattern: str) -> Path:
    matches = sorted(INPUT.rglob(pattern))
    if not matches:
        raise FileNotFoundError(f"{pattern} not found under {INPUT}. Attach the inputs listed at the top of this notebook.")
    return matches[0]


def run(*args) -> None:
    command = [sys.executable, "-u", *map(str, args)]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        raise RuntimeError(f"command failed with exit code {process.returncode}")


CODE = find_input("wikigpt/__init__.py").parents[1]
sys.path.insert(0, str(CODE))
sys.dont_write_bytecode = True
os.environ.update(PYTHONPATH=str(CODE), PYTHONDONTWRITEBYTECODE="1", HF_HUB_DISABLE_PROGRESS_BARS="1")

print("code:", CODE)
print("inputs:", ", ".join(sorted(p.name for p in INPUT.iterdir())))
print("torch:", torch.__version__)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print(f"free disk: {shutil.disk_usage(WORKING).free / 1e9:.1f} GB")

## Tests

In [ ]:
run("-m", "pytest", CODE / "tests", "-q", "-p", "no:cacheprovider")

## Prepare data

Expect roughly 30 to 90 minutes. If the cell stops part way, run it again: finished shards are skipped.

In [ ]:
DATA = WORKING / "data"
run(CODE / "scripts" / "prepare_data.py", "--out", DATA)

## Check

In [ ]:
import json

from wikigpt.tokenizer import Tokenizer

meta = json.loads((DATA / "meta.json").read_text())
print(json.dumps(meta, indent=2))

tokenizer = Tokenizer.load(DATA / "tokenizer.json")
sample = "The Eiffel Tower is a wrought-iron lattice tower in Paris, France. It was completed in 1889."
ids = tokenizer.encode(sample)
print(f"{len(sample)} characters -> {len(ids)} tokens")
print([tokenizer.decode([i]) for i in ids])
assert tokenizer.decode(ids) == sample
assert (DATA / "train.bin").stat().st_size == meta["train_tokens"] * 2